In [ ]:
from utils import train_load_datasets_resnet as tr
from torch.utils.data import DataLoader, ConcatDataset
from torch.utils.data import random_split
from torchvision import transforms
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
flag = 'organamnist'
color = False  # Colors for the flags
use_randaugment = True         # <- enable/disable RandAugment here
randaugment_ops = 2            # number of ops per image
randaugment_mag = 9            # magnitude (0-10 typical)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
size = 224  # Image size for the models
batch_size = 128

if color is True:
    train_tfms = []
    if use_randaugment:
        train_tfms.append(transforms.RandAugment(num_ops=randaugment_ops, magnitude=randaugment_mag))
    train_tfms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
    ]
    transform_train = transforms.Compose(train_tfms)

    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
    ])
    
else:
    train_tfms = []
    if use_randaugment:
        train_tfms.append(transforms.RandAugment(num_ops=randaugment_ops, magnitude=randaugment_mag))
    train_tfms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5], std=[.5]),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1))
    ]
    transform_train = transforms.Compose(train_tfms)

    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5], std=[.5]),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1))
    ])

# Load plain datasets/loaders (no augmentation)
[train_dataset_plain, calibration_dataset, test_dataset], [_, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform_eval, batch_size)

if use_randaugment:
    print(f'Using RandAugment with {randaugment_ops} ops and magnitude {randaugment_mag}')
    # Build an augmented view aligned to the same 80% train indices
    # 1) Recreate train+val with augmented transform
    aug_triplet, _ = tr.get_datasets(flag, im_size=size, color=color, transform=transform_train)
    combined_aug = ConcatDataset([aug_triplet[0], aug_triplet[1]])

    # 2) Wrap with the exact same indices as the 80% train subset
    train_dataset_aug = torch.utils.data.Subset(combined_aug, train_dataset_plain.indices)

    train_loaders, val_loaders = tr.CV_train_val_loaders(train_dataset_aug, train_dataset_plain, batch_size=batch_size)
else:
    print('Not using RandAugment')
    train_loaders, val_loaders = tr.CV_train_val_loaders(None, train_dataset_plain, batch_size=batch_size)

In [ ]:
models = []
for i in range(5):
    print('MODEL ' + str(i))
    model = tr.train_resnet18(
        flag,
        train_loader=train_loaders[i],
        val_loader=val_loaders[i],
        test_loader=test_loader,
        num_epochs=2,    
        learning_rate=0.0001,
        device='cuda:0',
        #random_seed=42  # Set a fixed seed for reproducibility
    )
    models.append(model)

In [ ]:
tr.evaluate_model(model=models, test_loader=test_loader, data_flag=flag, device='cuda:0')

In [ ]:
for i, model in enumerate(models):
    tr.save_model(model, path=f'models/224x224/resnet18_augmented_{flag}_224_{i}.pt')